# Data processing with EODHD

## Libraries

In [ ]:
import requests                 # For API data requests
import pandas as pd             # Data manipulation
import time                     # For maintaning good API conduct
import numpy as np              # Extended calculation and mathematical tools
import yfinance as yf           # Currency pairs and brent oil futures data
from tqdm.notebook import tqdm  # Progress bar for requests


api_key = "" # Insert your API-key here
exchange = "OL"
end_date = "2026-01-01"
start_date = "2018-01-01"

# Collecting and cleaning the data universe

## Unadjusted close prices for listed and delisted companies:
The unadjusted close price will be used for feature generation. The reason is that adjusted prices include lookahead-bias as they incorporate future corporate events that were not known in the past. In other words, it breaks the conditional information set, which incorporates information only at the point-in-time. 

In [ ]:
def get_all_historical_tickers(api_key):
    """
    Combines active and delisted tickers to mitigate survivorship bias.
    """

    # fetches active tickers as of now.
    active_url = f'https://eodhd.com/api/exchange-symbol-list/OL'
    active_params = {"api_token": api_key, "fmt": "json"}
    active_data = requests.get(active_url, params=active_params).json()

    # fetches all delisted tickers through time
    delisted_url = f'https://eodhd.com/api/exchange-symbol-list/OL'
    delisted_params = {"api_token": api_key, "fmt": "json", "delisted": 1}
    delisted_data = requests.get(delisted_url, params=delisted_params).json()
    
    # Combine lists
    full_list = active_data + delisted_data
    
    valid_tickers = []
    for item in full_list:
        # Filter for Common Stocks
        if item.get("Type") == "Common Stock":
            # If delisted, check if it was still around after our start_date
            # EODHD provides 'IsDelisted' and 'DelistedDate' in some formats
            # but usually, we just pull them and let the EOD check handle it
            valid_tickers.append(item["Code"])
            
    return list(set(valid_tickers)) # Remove duplicates

# Fetch close price data
def get_hist_unadjusted_close(api_key, ticker, start_date, end_date):

    symbol = f"{ticker}.{exchange}"
    url = f"https://eodhd.com/api/eod/{symbol}"
    params = {
        "api_token": api_key,
        "from": start_date,
        "to": end_date,
        "period": 'd', # daily observations
        "fmt": 'json'    
    }

    try:
        response = requests.get(url, params)
        if response.status_code == 200:
            data = response.json()
            if not data:
                return None
            
            df = pd.DataFrame(data)
            df["date"] = pd.to_datetime(df["date"])

            df = df[["date", "close"]].set_index("date")
            df.columns = [ticker]

            return df
        else:
            print(f"failed to fetch {symbol}: {response.status_code}")
    except Exception as e:
        print(f"error processing {ticker}: {e}")

# Main loop
def main():
    print("Fetching constituents...")
    tickers = get_all_historical_tickers(api_key)
    print(f"{len(tickers)} tickers were found... initiating data retrieval")

    complete_data = []
    
    for ticker in tqdm(tickers, desc="Fetching historical unadjusted close prices", unit="ticker"): 
        print(f"downloading {ticker}...")
        df = get_hist_unadjusted_close(api_key, ticker, start_date, end_date)
        if df is not None:
            complete_data.append(df)
        
        time.sleep(0.1) # brief sleep for API conduct

    if complete_data:
        final_df = pd.concat(complete_data, axis = 1)
        print(f"retrieval complete.")
        final_df.to_csv("unadj_close_OSEBX_listed_and_delisted.csv")
        print(f"Saving to CSV file in pre-located folder")
    else:
        print("No data retrieved")

main()

In [ ]:
close_prices = pd.read_csv("unadj_close_OSEBX_listed_and_delisted.csv", index_col=0, parse_dates=True)
close_prices.head()

## Adjusted close prices for listed and delisted companies:
The adjusted close will be used for calculating returns, and will serve as the dependent variable in all factor models.

In [ ]:
def get_all_historical_tickers(api_key):
    """
    Combines active and delisted tickers to mitigate survivorship bias.
    """

    active_url = f'https://eodhd.com/api/exchange-symbol-list/OL'
    active_params = {"api_token": api_key, "fmt": "json"}
    active_data = requests.get(active_url, params=active_params).json()

    delisted_url = f'https://eodhd.com/api/exchange-symbol-list/OL'
    delisted_params = {"api_token": api_key, "fmt": "json", "delisted": 1}
    delisted_data = requests.get(delisted_url, params=delisted_params).json()
    
    # Combine lists
    full_list = active_data + delisted_data
    
    valid_tickers = []
    for item in full_list:
        # Filter for Common Stocks
        if item.get("Type") == "Common Stock":
            # If delisted, check if it was still around after our start_date
            # EODHD provides 'IsDelisted' and 'DelistedDate' in some formats
            # but usually, we just pull them and let the EOD check handle it
            valid_tickers.append(item["Code"])
            
    return list(set(valid_tickers)) # Remove any duplicates

# Fetch close price data
def get_hist_adj_close(api_key, ticker, start_date, end_date):

    symbol = f"{ticker}.{exchange}"
    url = f"https://eodhd.com/api/eod/{symbol}"
    params = {
        "api_token": api_key,
        "from": start_date,
        "to": end_date,
        "period": 'd',
        "fmt": 'json'    
    }

    try:
        response = requests.get(url, params)
        if response.status_code == 200:
            data = response.json()
            if not data:
                return None
            
            df = pd.DataFrame(data)
            df["date"] = pd.to_datetime(df["date"])

            df = df[["date", "adjusted_close"]].set_index("date")
            df.columns = [ticker]

            return df
        else:
            print(f"failed to fetch {symbol}: {response.status_code}")
    except Exception as e:
        print(f"error processing {ticker}: {e}")

def main():
    print("Fetching constituents...")
    tickers = get_all_historical_tickers(api_key)
    print(f"{len(tickers)} tickers were found... initiating data retrieval")

    complete_data = []
    
    for ticker in tqdm(tickers, desc="Fetching historical adjusted close prices", unit="ticker"): 
        print(f"downloading {ticker}...")
        df = get_hist_adj_close(api_key, ticker, start_date, end_date)
        if df is not None:
            complete_data.append(df)
        
        time.sleep(0.1) # Brief sleep for good API conduct

    if complete_data:
        final_df = pd.concat(complete_data, axis = 1)
        print(f"retrieval complete.")
        final_df.to_csv("adj_close_OSEBX_listed_and_delisted.csv")
        print(f"Saving to CSV file in pre-located folder")
    else:
        print("No data retrieved")

main()

In [ ]:
adj_close_prices = pd.read_csv("adj_close_OSEBX_listed_and_delisted.csv", index_col=0, parse_dates=True)
adj_close_prices.head()

In [ ]:
prices_diff = adj_close_prices.sub(close_prices)
desc = prices_diff.describe()

filtered_desc = desc.loc[:, desc.loc["mean"] == 0]
filtered_desc.columns

## Identify companies with no balance sheet data

In [ ]:
ticker_no_bs = []

def fetch_bs(api_key, ticker, exchange) -> pd.DataFrame:
     """
     Identify tickers with no balance sheet data available.
     """
     url =  f"https://eodhd.com/api/fundamentals/{ticker}.{exchange}"
     params = {
        "api_token": api_key,
        "fmt": "json",
        "filter": "Financials"
        }

     try:
         response = requests.get(url, params=params)
         if response.status_code != 200:
              print(f"{ticker} HTTP error: {response.status_code}")
              return None
         
         data = response.json()
         
         quarterly_bs = (
              data.get("Balance_Sheet", {})
              .get("quarterly", {})
              )
         
         if not quarterly_bs:
              return ticker
         else:
              return None
         
     except Exception as e:
          print(f"{ticker}: error in identifying balance sheet... exception: {e}")
          return ticker

for ticker in tqdm(close_prices.columns, desc="identifying balance sheets...", unit="ticker"):
    print(f"Processing {ticker}...")
    val = fetch_bs(api_key, ticker, exchange)
    if val != None:
        ticker_no_bs.append(fetch_bs(api_key, ticker, exchange))
    time.sleep(0.1) 
         
print("Tickers with no balance sheet data have been retrieved")

# Save removeable tickers
tick_remove = pd.Series(ticker_no_bs)
tick_remove.to_csv("ticker_removal(no_balance_sheet).csv")

## Inspect metadata and clean price data

In [ ]:
def get_metadata(delisted=int):
    url = f"https://eodhd.com/api/exchange-symbol-list/{exchange}"

    params = {
        "api_token": api_key,
        "fmt": "json",
        "delisted": delisted
    }

    try:
        response = requests.get(url, params=params)

        if response.status_code == 200:
            data = response.json()
            return pd.DataFrame(data)
        else:
            print(f"error fetching exchange and symbols metadata")
            e = response.status_code
            return None
    except Exception as e:
        print(f"Error status code: {e}")

def merge_data():

    listed = get_metadata(0)
    delisted = get_metadata(1)

    return pd.concat((listed, delisted), axis=0)

metadata = merge_data()
metadata = metadata[(metadata["Type"] == "Common Stock") & (metadata["Code"].isin(close_prices.columns))]
metadata.to_csv("metadata.csv")

In [ ]:
# Remove old merkur tickers
metadata = pd.read_csv("metadata.csv", index_col=0)
old_merkur = list(metadata["Code"][metadata["Code"].str.endswith("-ME")].values)
# These are essentially duplicates of more recent Euronext Growth ticker names found in the data, which are noted without "-ME".

# Define removal list
tick_remove_no_bs = pd.read_csv("ticker_removal(no_balance_sheet).csv", index_col=0)["0"].to_list()
remove = pd.read_csv("remove_tickers_claude_revision.csv")["0"].values.tolist()
remove.extend(["SCHA", "SCHB", "SCOIN", "STSU", "KAL", "BSP", "AWDR", "AINMT", "DDRIL", "EAM"]) # Outliers 
remove.extend(tick_remove_no_bs)
remove.extend(old_merkur)
remove = list(set(remove))
remove.remove("DLTXO")

# drop from prices
close_prices = pd.read_csv("unadj_close_OSEBX_listed_and_delisted.csv", index_col=0, parse_dates=True)
close_prices = close_prices.drop(columns=remove)

adj_close_prices = pd.read_csv("adj_close_OSEBX_listed_and_delisted.csv", index_col=0, parse_dates=True)
adj_close_prices = adj_close_prices.drop(columns=remove)

# Combine old and new tickers (is treated as seperate entities by EODHD, without overlapping price history)
close_prices["UNIV"] = close_prices["UNIV"].combine_first(close_prices["DLTXO"]) # Same company, but with name change
adj_close_prices["UNIV"] = adj_close_prices["UNIV"].combine_first(adj_close_prices["DLTXO"]) # Same company, but with name change

close_prices = close_prices.drop(columns="DLTXO")
adj_close_prices = adj_close_prices.drop(columns="DLTXO")

In [ ]:
# Calculate missing percentage
missing_pct = close_prices.isnull().sum() / len(close_prices)

# Identify tickers to drop
tickers_to_drop = missing_pct[missing_pct > 0.90].index.tolist()
df_clean = close_prices.drop(columns=tickers_to_drop)

# see what was dropped
print(f"Removed {len(tickers_to_drop)} tickers: {tickers_to_drop}")

In [ ]:
# Calculate missing percentage
missing_pct = adj_close_prices.isnull().sum() / len(adj_close_prices)

# Identify tickers to drop
tickers_to_drop = missing_pct[missing_pct > 0.90].index.tolist()
df_clean = adj_close_prices.drop(columns=tickers_to_drop)

# see what was dropped
print(f"Removed {len(tickers_to_drop)} tickers: {tickers_to_drop}")

In [ ]:
# Format remove tickers from both adj- and unadj close prices
final_adj_close_prices = adj_close_prices.drop(columns=tickers_to_drop)
final_adj_close_prices = final_adj_close_prices[sorted(final_adj_close_prices.columns)]
final_adj_close_prices.to_csv("final_adj_close_prices.csv")

close_prices = close_prices[sorted(close_prices.columns)]
common_tickers = close_prices.columns.intersection(final_adj_close_prices.columns) # removing the same columns in unadj close prices as in adj close prices
final_close_prices = close_prices[common_tickers]
final_close_prices.to_csv("final_close_prices.csv")

In [ ]:
close_prices = pd.read_csv("final_close_prices.csv", index_col=0)
close_prices.index = pd.to_datetime(close_prices.index)
close_prices

## Identify excessive correlation

In [ ]:
# the unadjusted close prices is used so that it reveals simultanous corporate events
close_prices = pd.read_csv("final_close_prices.csv", index_col=0)
close_prices.index = pd.to_datetime(close_prices.index)

returns = close_prices.pct_change()
corr_matrix = returns.corr()

# Convert the correlation matrix into a multi series format
corr_matrix = corr_matrix - np.eye(len(corr_matrix)) # Sorting works better when asset correlation is not paired to themselves
pairs = corr_matrix.unstack()

# Investigate correlations greater than 0.9
high_corr = pairs[abs(pairs) > 0.90]

# Sort values and remove duplicates
suspects = high_corr.sort_values(ascending=False).drop_duplicates()

print(suspects)

In the case where some tickers are old names, we present the old name and new ticker symbol in the respective order.
- SOHO is LUMI group, Sonans holding ticker change.
- AYFIE is AIX.
- SCHA and SCHB (Split share rights into A and B series) is VEND
- CARBN is the old ticker of AQUIL, and  AQUIL is the old ticker for DFENS.
- STSU is SCOIN.
- TRVX is CRNA. 
- CSAM is OMDA.
- SIOFF is SEA1.
- ELOP is ININ.
- BERGEN is ENDUR.
- ULTI and ZLNA was a merger (ULTI -> ZLNA).
- R8P is RCR.
- BGBIO and ONCIN was a merger, however they will be treated differently due to non-overlapping balance-sheet data.
- AEGA was NOFIN, and then NOFIN again. (AEGA -> NOFIN)
- SBX is ENH.
- IFISH is KLDVK.
- B2H is B2I.
- SADG is ROGS.
- BRA and LEA was a merger (BRA -> LEA).

SEAW7 and TEAM is inversely perfectly by chance, they overlap on 2 observations.

In [ ]:
# It is identified from personal inspection that data has not been carried over from previous ticker names
overlap_fundamentals_merge = {
    "SOHO": "LUMI", 
    "AYFIE": "AIX",
    "TRVX": "CRNA",
    "B2H": "B2I",
    "BRA": "LEA",
    "CSAM": "OMDA",
    "CARBN": "AQUIL", 
    "AQUIL": "DFENS",
    "SIOFF": "SEA1",
    "ELOP": "ININ", 
    "BERGEN": "ENDUR", 
    "ULTI": "ZLNA", 
    "R8P": "RCR", 
    "SBX": "ENH",
    "BGBIO": "ONCIN", 
    "IFISH": "KLDVK",
    "SADG": "ROGS",
    "PSKY": "CODE",
    "HMONY": "LOKO",
    "BEL": "BELCO",
    "NOR": "BNOR",
    "ITE": "ITERA",
    "NOFI": "BANO",
    "INFRNT": "INFRO",
    "SVEG": "SBNOR",
    "NANO": "NANOV",
    "NANOV": "TRMED",
    "SALMON": "SACAM",
    "BDRILL": "BORR",
    "ARCHER": "ARCH"
    }

## Identify outliers/Faulty data

### Adjusted close

In [ ]:
adj_close_prices = pd.read_csv("final_adj_close_prices.csv", index_col=0)
adj_close_prices.index = pd.to_datetime(adj_close_prices.index)

returns = adj_close_prices.pct_change()

threshold = 1.0
outliers = returns.stack()[returns.stack().abs() > threshold]
outliers = outliers.reset_index()
outliers.columns = ["date", "ticker", "return"]
outliers = outliers.sort_values("return", key=abs, ascending=False)
outliers["return_pct"] = (outliers["return"] * 100).round(1).astype(str) + "%"
print(f"Total outlier events: {len(outliers)}")
print(f"Unique tickers: {outliers['ticker'].nunique()}")
print(outliers[["date", "ticker", "return_pct"]].to_string(index=False))

- NOFIN, formerly AEGA is apparantly a legit price discovery event.
- Yet many other events have failed to be accounted for in ajdusted data, we therefore drop it. 

In [ ]:
repeat_offenders = outliers.groupby("ticker").filter(lambda x: len(x) >= 2)
print(repeat_offenders["ticker"].value_counts())
print(repeat_offenders[["date", "ticker", "return_pct"]].to_string(index=False))

### Unadjusted close

In [ ]:
close_prices = pd.read_csv("final_close_prices.csv", index_col=0)
close_prices.index = pd.to_datetime(close_prices.index)

returns = adj_close_prices.pct_change()

threshold = 1.0
outliers = returns.stack()[returns.stack().abs() > threshold]
outliers = outliers.reset_index()
outliers.columns = ["date", "ticker", "return"]
outliers = outliers.sort_values("return", key=abs, ascending=False)
outliers["return_pct"] = (outliers["return"] * 100).round(1).astype(str) + "%"
print(f"Total outlier events: {len(outliers)}")
print(f"Unique tickers: {outliers['ticker'].nunique()}")
print(outliers[["date", "ticker", "return_pct"]].to_string(index=False))

Can not find proof of the outlier realizations for: STSU, SCOIN, KAL, BSP, AWDR, AINMT, DDRIL, EAM.
- STSU and SCOIN is seemingly a corrput time-series not reflecting any true price data.
- It is not unrealistic to assume that these have broken data.

In [ ]:
outlier_remove = ["STSU", "SCOIN", "KAL", "BSP", "AWDR", "AINMT", "DDRIL", "EAM"]

# mask = [outlier in outlier_remove for outlier in remove]
# print(f"{sum(mask)} out of {len(outlier_remove)} outliers in previous removal process")

## Sample with listing and delisting

In [ ]:
# Load your data (ensure the index is converted to datetime)
book = pd.read_csv("book_equity.csv", index_col=0).dropna(how="all", axis=1)
book.index = pd.to_datetime(book.index)

close_prices = pd.read_csv("final_close_prices.csv", index_col=0)[book.columns]
close_prices.index = pd.to_datetime(close_prices.index)


df = close_prices
df.index = pd.to_datetime(df.index)

# Create a summary table
status_report = []

for ticker in df.columns:
    first_date = df[ticker].first_valid_index()
    last_date = df[ticker].last_valid_index()
    
    status_report.append({
        'Ticker': ticker,
        'First Listing Date': pd.to_datetime(first_date),
        'Last Available Date': pd.to_datetime(last_date),
        'Is Active': last_date == df.index.max() # True if data exists on the most recent date
    })

report_df = pd.DataFrame(status_report)

# Sort by First Listing Date to see recent IPOs
report_df.sort_values(by='First Listing Date', ascending=False)

## Survival metadata

In [ ]:
inactive = report_df["Ticker"][report_df["Is Active"] == False].tolist()
active = report_df["Ticker"][report_df["Is Active"] == True].tolist()

active_whole_period = report_df[(report_df["First Listing Date"] == "2018-01-02") & (report_df["Is Active"])].sort_values(by="Last Available Date")
died_through_period = report_df[(report_df["First Listing Date"] == "2018-01-02") & (~report_df["Is Active"])].sort_values(by="Last Available Date")
late_entries_survived = report_df[(report_df["First Listing Date"] > "2018-01-02") & (report_df["Is Active"])].sort_values(by="Last Available Date")
late_entries_not_survived = report_df[(report_df["First Listing Date"] > "2018-01-02") & (~report_df["Is Active"])].sort_values(by="Last Available Date")

print(f"Out of {len(active) + len(inactive)} tickers, and {len(active)} ticker are active to {end_date}")
print(f"{len(active_whole_period)} tickers have survived the whole period.")
print(f"{len(died_through_period)} tickers have not survived the whole period.")
print(f"Tickers with late entries that have survived is {len(late_entries_survived)}.")
print(f"Tickers with late entries that have not survived is {len(late_entries_not_survived)}.")

# Currency adjustments

## Time-series currency denotation meta data
- It is seemingly so that all common stock are listed in the Norwegian currency NOK through the exchange OL in their time-series prices
- Even though it is seeming it may not be the case. However, upon closer and personal inspection of missing values, the possibility of other currency bases do not reveal themselves.

In [ ]:
def get_metadata(delisted=int):
    url = f"https://eodhd.com/api/exchange-symbol-list/{exchange}"

   
    params = {
        "api_token": api_key,
        "fmt": "json",
        "delisted": delisted
    }

    try:
        response = requests.get(url, params=params)

        if response.status_code == 200:
            data = response.json()
            return pd.DataFrame(data)
        else:
            print(f"error fetching exchange and symbols metadata")
            e = response.status_code
            return None
    except Exception as e:
        print(f"Error status code: {e}")

def merge_data():

    listed = get_metadata(0)
    delisted = get_metadata(1)

    return pd.concat((listed, delisted), axis=0)

metadata = merge_data()
metadata = metadata[metadata["Type"] == "Common Stock"]
metadata.to_csv("metadata.csv")

In [ ]:
# No unique currency other than NOK
metadata["Currency"].unique()

## Report currency

In [ ]:
def get_balance_sheet_currency(api_key, ticker):
    """
    Retrieves the reporting currency from the Balance Sheet section
    of the EODHD fundamentals endpoint for a single ticker.
    """
    url = f"https://eodhd.com/api/fundamentals/{ticker}.{exchange}"
    params = {
        "api_token": api_key, 
        "fmt": "json"
        }
    
    try:
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print(f"{ticker}: HTTP error {response.status_code}")
            return None
        
        data = response.json()
        currency = (data.get("Financials", {})
                        .get("Balance_Sheet", {})
                        .get("currency_symbol", None))
        return currency
    
    except Exception as e:
        print(f"{ticker}: Error - {e}")
        return None
# Retrieve currency for all tickers

tickers = close_prices.columns
currency_map = {}

for ticker in tqdm(tickers, desc="Fetching ticker report currency...", unit="ticker"):
    print(f"Fetching currency for {ticker}...")
    currency_map[ticker] = get_balance_sheet_currency(api_key, ticker)
    time.sleep(0.1)
 

currency_series = pd.Series(currency_map, name="currency_symbol")
currency_series.to_csv("ticker_currency_map.csv")
print("Retrieval complete")

In [ ]:
currency_series = pd.read_csv("ticker_currency_map.csv", index_col=0)

In [ ]:
currency_series.loc["NOM"]

NOM is the only ticker missing a balance sheet currency, upon personal inspection, NOM's finanical report are denoted in NOK

In [ ]:
currency_series.loc["NOM"] = "NOK"
currency_series.to_csv("final_currency_series.csv")

In [ ]:
currency_series[currency_series.isna().values]

Missing balance sheet currencies (found with personal inspection of published reports):
- CAPSL: NOK
- GEM: NOK
- HAFNI: USD

In [ ]:
currency_series.loc["CAPSL",:] = "NOK"
currency_series.loc["GEM",:] = "NOK"
currency_series.loc["HAFNI",:] = "NOK"

currency_series.isna().sum()
currency_series.to_csv("final_currency_series.csv")

## Retrieve and map historical pairs

In [ ]:
currency_series = pd.read_csv("final_currency_series.csv", index_col=0)

curr_symbols  = currency_series["currency_symbol"].unique().tolist()
curr_symbols.remove("NOK")

def retrieve_pairs(curr_symbols, start, end):
    """
    Retrieve historical currency pairs, and map correct conversion rates.
    """
    pair_symbols = {i: f"{i}NOK=X" for i in curr_symbols}

    # USD pair for NOK in yahoo finance is just denoted "NOK=X"
    pair_symbols["USD"] = "NOK=X"
    pair_symbols["SEK"] = "NOKSEK=X" # NOKSEK=X is better behaved for database queries. 
    pairs = list(pair_symbols.values())

    df = yf.download(pairs, start, end)["Close"]

    map = {i: j for j, i in pair_symbols.items()}
    df = df.rename(columns=map)

    df["SEK"] = 1 / df["SEK"] # Correction for conversion

    df["NOK"] = 1.0           # This column is meant to make control flow easier.

    return df, pair_symbols

fx_start = "2015-01-01"

pairs, pair_symbols = retrieve_pairs(curr_symbols, fx_start, end_date)
pairs.to_csv("historical_currency_pairs.csv")
pd.Series(pair_symbols).to_csv("pair_symbols.csv")

In [ ]:
pair_symbols

# Feature engineering

### Fundamental data call
The purpose of this code segment is to retrieve various fundamental data that are composites of our included factors.

In [ ]:
# Maps currency to company report
currency_series = pd.read_csv("final_currency_series.csv", index_col=0)

# Time-series of historical currency pair values
pairs = pd.read_csv("historical_currency_pairs.csv", index_col=0)
pairs.index = pd.to_datetime(pairs.index)

# Close prices are used for requesting data for relevant tickers
close_prices = pd.read_csv("final_close_prices.csv", index_col=0)
close_prices.index = pd.to_datetime(close_prices.index)


def safe_ts(x):
    """
    Date formatting function for creating time-series of fundamental reporting values
    -------
    """
    try:
        ts = pd.to_datetime(x)
        return ts.normalize() if pd.notnull(ts) else pd.NaT
    except (ValueError, TypeError):
        return pd.NaT

def extract_pit_mapping(earnings_history) -> dict:
    """
    Extracts report dates and fiscal dates
    ------
    """
    mapping = {}
    for _, e in earnings_history.items():
        fiscal = safe_ts(e.get("date"))
        report = safe_ts(e.get("reportDate"))
        if pd.notnull(fiscal) and pd.notnull(report):
            mapping[fiscal] = report
    return mapping


def snap_to_daily(dates, values, calendar):
    """
    Snaps values to daily frequency using LOCF.
    --------
    calendar : relevant calendar / list of datetimes
        Use the main time-series calendar for example time_series.index
    --------
    """
    s = pd.Series(values.values, index=pd.to_datetime(dates))
    idx = pd.DatetimeIndex(calendar).union(s.index).sort_values()
    return s.reindex(idx).ffill().reindex(calendar)

# Corrects fundamental values not denominated in NOK to NOK
def get_nok_per_unit(fx_rates: pd.DataFrame, currency: str, date: pd.Timestamp) -> float:
    if currency == "NOK":
        return 1.0
    if currency not in fx_rates.columns:
        raise KeyError(f"Warning: Currency '{currency}' not in pairs data.")
    available = fx_rates[currency].loc[:date].dropna()
    if available.empty:
        raise ValueError(f"No FX rate for '{currency}' on or before {date.date()}")
    return available.iloc[-1]



def fetch_fundamentals_strict(api_key, ticker, exchange, currency_series, pairs, start_date, end_date):
    """
    Fetch fundamental data from tickers from EODHD.
    -------
    This function fetches a variety of values obtained from the financial reports of firms to be using in predictive modeling. 
    The reason why it is called strict is due to the removal of data where publishing dates can not be retrieved, and the data can therefore not be PIT-adjusted.

    The function returns historical values of book equity, total asset value, total liabilities (all denoted in a report respective currency), and shares outstanding (reported in units). 
    The function requires several variables for working.

    -------
    api_key : API key from data provider EODHD, requires EODHD "All-In-One" package.

    ticker : str
        ticker name

    exchange : str
        exhange symbol specifically "OL"

    currency_series : pd.Series

    pairs : pd.DataFrame

    start_date & end_date : str or pd.Timestamp - "%Y-%m-%d"
        Period for data retrieval.
    """

    url = f"https://eodhd.com/api/fundamentals/{ticker}.{exchange}"
    params = {"api_token": api_key, "fmt": "json"}

    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)
    fx_start = pd.Timestamp("2017-07-01")

    try:
        response = requests.get(url, params=params, timeout=30)
        if response.status_code != 200:
            return None

        data = response.json()
        currency = currency_series.loc[ticker, "currency_symbol"]

        bs_q = data.get("Financials", {}).get("Balance_Sheet", {}).get("quarterly", {})
        earnings = data.get("Earnings", {}).get("History", {})

        if not bs_q or not earnings:
            return None

        pit_map = extract_pit_mapping(earnings)
        records = []

        for _, rep in bs_q.items():
            fiscal = safe_ts(rep.get("date"))

            if fiscal not in pit_map:
                continue

            report_date = pit_map[fiscal]

            if report_date > end_date or fiscal < fx_start:
                continue

            equity = rep.get("totalStockholderEquity")
            shares = rep.get("commonStockSharesOutstanding")
            assets = rep.get("totalAssets")
            liab   = rep.get("totalLiab")

            if None in (equity, shares, assets, liab) or shares == 0:
                continue

            fx = get_nok_per_unit(pairs, currency, fiscal)

            records.append({
                "report_date": report_date,
                "fiscal_date": fiscal,
                "book_equity": float(equity) * fx,
                "shares": float(shares),
                "assets": float(assets) * fx,
                "liabilities": float(liab) * fx,
            })

        if not records:
            return None

        df = pd.DataFrame(records).sort_values("report_date").drop_duplicates("report_date")
        df = df[df["report_date"] <= end_date]

        # Detect and correct thousand-unit outliers (EODHD has in very few cases reported numbers in thousands)
        # The loop aims to correct
        asset_median = df["assets"].median()
        for col in ["assets", "book_equity", "liabilities"]:
            ratio = df[col] / asset_median
            # If value is ~1000x smaller than median, it's in thousands
            df[col] = df[col].where(ratio > 0.01, df[col] * 1000)
            
        return df

    except Exception as e:
        print(f"Failed retrieval for {ticker}: {e}")
        return None



close_prices = pd.read_csv("final_close_prices.csv", index_col=0, parse_dates=True)
tickers = close_prices.columns
calendar = pd.to_datetime(close_prices.index)

# Initalize values
book_d = {}
shares_d = {}
assets_d = {}
liab_d = {}
inclusion_mask = {}
growth_d = {}

# Main request loop
for ticker in tqdm(tickers, desc="Fetching fundamentals"):


    df = fetch_fundamentals_strict(
        api_key, ticker, exchange,
        currency_series, pairs,
        start_date, end_date
    )

    # Insert NaN where None is found.
    if df is None:
        book_d[ticker] = pd.Series(np.nan, index=calendar)
        shares_d[ticker] = pd.Series(np.nan, index=calendar)
        assets_d[ticker] = pd.Series(np.nan, index=calendar)
        liab_d[ticker] = pd.Series(np.nan, index=calendar)
        inclusion_mask[ticker] = pd.Series(False, index=calendar)
        growth_d[ticker] = pd.Series(np.nan, index=calendar)
        continue


    assert (df["fiscal_date"].diff().dropna() > pd.Timedelta(0)).all() # Safety for ensuring chronological ordering.

    # Snap quarterly values from when they were published to daily values LOCF.
    book_d[ticker] = snap_to_daily(df["report_date"], df["book_equity"], calendar)
    shares_d[ticker] = snap_to_daily(df["report_date"], df["shares"], calendar)
    assets_d[ticker] = snap_to_daily(df["report_date"], df["assets"], calendar)
    liab_d[ticker] = snap_to_daily(df["report_date"], df["liabilities"], calendar)

    df["asset_growth"] = df["assets"].pct_change()
    growth_d[ticker] = snap_to_daily(df["report_date"], df["asset_growth"], calendar)

    inc = pd.Series(False, index=calendar)
    for _, row in df.iterrows():
        inc.loc[row["report_date"]:] = True
    inclusion_mask[ticker] = inc

    print(f"Retrieved fundamental data for {ticker}")

    time.sleep(0.2)


# Output data frames 
book_equity_df = pd.DataFrame(book_d)
shares_df = pd.DataFrame(shares_d)
assets_df = pd.DataFrame(assets_d)
liabilities_df = pd.DataFrame(liab_d)
inclusion_mask_df = pd.DataFrame(inclusion_mask)
asset_growth_df = pd.DataFrame(growth_d)

# Some ticker symbol changes have failed to carry over fundamental data for some tickers,
# These are the ones identifed:
# Dictionaries preserve insertion ordering (e.g. NANO -> NANOV -> TRMED) for multiple name changes.
overlap_fundamentals_merge = {
    "SOHO": "LUMI", 
    "AYFIE": "AIX",
    "TRVX": "CRNA",
    "B2H": "B2I",
    "BRA": "LEA",
    "CSAM": "OMDA",
    "CARBN": "AQUIL", 
    "AQUIL": "DFENS",
    "SIOFF": "SEA1",
    "ELOP": "ININ", 
    "BERGEN": "ENDUR", 
    "ULTI": "ZLNA", 
    "R8P": "RCR", 
    "SBX": "ENH",
    "BGBIO": "ONCIN", 
    "IFISH": "KLDVK",
    "SADG": "ROGS",
    "PSKY": "CODE",
    "HMONY": "LOKO",
    "BEL": "BELCO",
    "NOR": "BNOR",
    "ITE": "ITERA",
    "NOFI": "BANO",
    "INFRNT": "INFRO",
    "SVEG": "SBNOR",
    "NANO": "NANOV",
    "NANOV": "TRMED",
    "SALMON": "SACAM",
    "BDRILL": "BORR",
    "ARCHER": "ARCH",
    "AEGA" : "NOFIN"
    }

# The solution is to overlap old and new tickers, while naming the fundmental column after the most recent ticker symbol.
def overlap_merge(df, merge_map):
    for old, new in merge_map.items():
        if old not in df.columns or new not in df.columns:
            continue
        df[new] = df[new].fillna(df[old])
        df.drop(columns=old, inplace=True)
    return df

book_equity_df = overlap_merge(book_equity_df, overlap_fundamentals_merge)
shares_df = overlap_merge(shares_df, overlap_fundamentals_merge)
assets_df = overlap_merge(assets_df, overlap_fundamentals_merge)
liabilities_df = overlap_merge(liabilities_df, overlap_fundamentals_merge)
asset_growth_df = overlap_merge(asset_growth_df, overlap_fundamentals_merge)
inclusion_mask_df = overlap_merge(inclusion_mask_df, overlap_fundamentals_merge)

print(f"Tickers after overlap merge: {len(book_equity_df.columns)}")

book_equity_df.to_csv("book_equity.csv")
shares_df.to_csv("shares.csv")
assets_df.to_csv("assets.csv")
liabilities_df.to_csv("liabilities.csv")
inclusion_mask_df.to_csv("inclusion_mask.csv")
asset_growth_df.to_csv("growth.csv")

print("Retrieval complete")
print(f"Tickers with data: {inclusion_mask_df.any().sum()}/{len(tickers)}")

In [ ]:
no_data_tickers = inclusion_mask_df.columns[inclusion_mask_df.sum() == 0].tolist()
print(f"Tickers with no PIT-valid data: {len(no_data_tickers)}")
print(no_data_tickers)

## Book-to-Market

In [ ]:
close_prices = pd.read_csv("final_close_prices.csv", index_col=0, parse_dates=True)

book_equity_df = pd.read_csv("book_equity.csv", index_col=0)
book_equity_df.index = pd.to_datetime(book_equity_df.index) 

shares_df = pd.read_csv("shares.csv", index_col=0)
shares_df.index = pd.to_datetime(shares_df.index) 

book_per_share = book_equity_df.div(shares_df, axis=1)

bm_per_share = book_per_share.div(close_prices, axis=1)

bm_per_share = bm_per_share.dropna(how="all", axis=1)

bm_per_share.to_csv("book-to-market(value).csv")

In [ ]:
bm_per_share[bm_per_share < 0].dropna(how="all", axis=1).to_csv("negative_bm_tickers.csv")

In [ ]:
bm_per_share[bm_per_share < 0].dropna(how="all", axis=1).isna().sum()

In [ ]:
bm_per_share["UNIV"][bm_per_share["UNIV"] < 0]

## Market cap
- Monthly
- Daily

In [ ]:
close_prices = pd.read_csv("final_close_prices.csv", index_col=0, parse_dates=True)
shares_df = pd.read_csv("shares.csv", index_col=0)
shares_df.index = pd.to_datetime(shares_df.index)

market_cap = shares_df.mul(close_prices, axis=1) # regular market cap

market_cap = market_cap.dropna(how="all", axis=1)

market_cap.to_csv("mcap.csv")

log_mcap = np.log(market_cap)

log_mcap.to_csv("log_mcap(size).csv")

## Rolling SQRT market cap 21 day mean

In [ ]:
mcap_filled = market_cap.ffill(limit=5)              
sqrt_mcap_smooth = np.sqrt(mcap_filled).rolling(21, min_periods=1).mean()
 
sqrt_mcap_smooth.to_csv("sqrt_mcap_smooth.csv")

## Illiquidity
### Retrieving daily trading volume

In [ ]:
# Fetch volume data
def get_hist_volume(api_key, ticker, start_date, end_date):

    symbol = f"{ticker}.{exchange}"
    url = f"https://eodhd.com/api/eod/{symbol}"
    params = {
        "api_token": api_key,
        "from": start_date,
        "to": end_date,
        "period": 'd',
        "fmt": 'json'    
    }

    try:
        response = requests.get(url, params)
        if response.status_code == 200:
            data = response.json()
            if not data:
                return None
            
            df = pd.DataFrame(data)
            df["date"] = pd.to_datetime(df["date"])

            df = df[["date", "volume"]].set_index("date")
            df.columns = [ticker]

            return df
        else:
            print(f"failed to fetch {symbol}: {response.status_code}")
    except Exception as e:
        print(f"error processing {ticker}: {e}")

close_prices = pd.read_csv("unadj_close_OSEBX_listed_and_delisted.csv", index_col=0, parse_dates=True)

def main():
    print("Fetching constituents...")
    tickers = close_prices.columns
    print(f"initiating data retrieval for {len(tickers)}")

    complete_data = []
    
    for ticker in tqdm(tickers, unit="ticker", desc="Fetching daily trading volume..."): 
        print(f"downloading volume for {ticker}...")
        df = get_hist_volume(api_key, ticker, start_date, end_date)
        if df is not None:
            complete_data.append(df)
        
        time.sleep(0.1) # Brief sleep for good API conduct

    if complete_data:
        final_df = pd.concat(complete_data, axis = 1)
        print(f"retrieval complete.")
        final_df.to_csv("daily_volume.csv")
        print(f"Saving to CSV file in pre-located folder")
    else:
        print("No data retrieved")

main()

In [ ]:
volume = pd.read_csv("daily_volume.csv", index_col=0, parse_dates=True)
print(f"UNIV obs {len(volume["UNIV"].dropna())}")
print(f"DLTXO obs {len(volume["DLTXO"].dropna())}")
print(f"Shared obs {len(volume[["UNIV","DLTXO"]].dropna())}")
volume["UNIV"] = volume["UNIV"].combine_first(volume["DLTXO"]) # Same company, but with name change

# Final historical volume dataframe
close_prices = pd.read_csv("final_close_prices.csv", index_col=0, parse_dates=True)
volume = volume[close_prices.columns]

volume.to_csv("volume.csv")

In [ ]:
adj_close_prices = pd.read_csv("final_adj_close_prices.csv", index_col=0, parse_dates=True)
close_prices = pd.read_csv("final_close_prices.csv", index_col=0, parse_dates=True)
returns = adj_close_prices.pct_change()
daily_volume = pd.read_csv("volume.csv", index_col=0, parse_dates=True)

nok_volume = daily_volume * close_prices
K = 252  # trailing 12 months

daily_illiquidity = (returns.abs() / nok_volume)
daily_illiquidity = daily_illiquidity.replace([np.inf, -np.inf], np.nan)


amihud_illiquidity = daily_illiquidity.rolling(K, min_periods=60).mean()



In [ ]:
amihud_illiquidity.isna().sum()

In [ ]:

threshold = 0.01
outliers = amihud_illiquidity.stack()[amihud_illiquidity.stack().abs() > threshold]
outliers = outliers.reset_index()
outliers.columns = ["date", "ticker", "value"]
outliers = outliers.sort_values("value", key=abs, ascending=False)
outliers["return_pct"] = (outliers["value"] * 100).round(1).astype(str) + "%"
print(f"Total outlier events: {len(outliers)}")
print(f"Unique tickers: {outliers['ticker'].nunique()}")
print(outliers[["date", "ticker", "value"]].to_string(index=False))

outliers[["date", "ticker", "value"]].to_csv("outlier_illiquidity.csv")

In [ ]:
amihud_illiquidity.to_csv("illiquidity.csv")

## Leverage 
### Debt-to-assets ratio

In [ ]:
liabilities_df = pd.read_csv("liabilities.csv", index_col=0)
liabilities_df.index = pd.to_datetime(liabilities_df.index) 

assets_df = pd.read_csv("assets.csv", index_col=0)
assets_df.index = pd.to_datetime(assets_df.index)

d_to_b = liabilities_df.div(assets_df, axis=1)

In [ ]:
d_to_b.describe()

In [ ]:

threshold = 2 # Twice as much debt as total assets
outliers = d_to_b.stack()[d_to_b.stack().abs() > threshold]
outliers = outliers.reset_index()
outliers.columns = ["date", "ticker", "value"]
outliers = outliers.sort_values("value", key=abs, ascending=False)
outliers["return_pct"] = (outliers["value"] * 100).round(1).astype(str) + "%"
print(f"Total outlier events: {len(outliers)}")
print(f"Unique tickers: {outliers['ticker'].nunique()}")
print(outliers[["date", "ticker", "value"]].to_string(index=False))

outliers[["date", "ticker", "value"]].to_csv("outlier_dtb.csv")

In [ ]:
d_to_b.to_csv("dtb(leverage).csv")

## Dividend yield

### Trailing 12-month Div. Yield

In [ ]:

close_prices = pd.read_csv("final_close_prices.csv", index_col=0, parse_dates=True)
close_prices.index = pd.to_datetime(close_prices.index)


def fetch_dividends(api_key, ticker, exchange, start_date, end_date):
    """
    Fetch PIT-adjusted dividend data from EODHD's dividend calendar.
    ------

    Dividends are PIT-adjusted by ex-date. LOCF is not used here. Here we use trailing dividends.
    """
    url = f"https://eodhd.com/api/div/{ticker}.{exchange}"
    params = {
        "api_token": api_key,
        "from": start_date,
        "to": end_date,
        "fmt": "json"
    }
    try:
        response = requests.get(url, params=params, timeout=30)
        if response.status_code != 200:
            return None
        data = response.json()
        if not data:
            return None
        records = []
        for d in data:
            ex = pd.to_datetime(d.get("date"))
            val = d.get("unadjustedValue")
            if pd.notnull(ex) and val is not None and float(val) > 0:
                records.append({"ex_date": ex, "div_per_share": float(val)})
        if not records:
            return None
        return pd.DataFrame(records)
    except Exception as e:
        print(f"Failed dividend fetch for {ticker}: {e}")
        return None


# Need dividends from 12 months before start_date for trailing window
lookback_start = pd.to_datetime(start_date) - pd.DateOffset(months=12)
end_dt = pd.to_datetime(end_date)

calendar = pd.to_datetime(close_prices.index)
tickers =  close_prices.columns

div_yield_d = {}

for ticker in tqdm(tickers, desc="Fetching dividends"):

    df = fetch_dividends(api_key, ticker, exchange, lookback_start, end_dt)

    if df is None:
        div_yield_d[ticker] = pd.Series(np.nan, index=calendar)
        continue

    # Place dividends on ex-dates, sum if multiple on same date
    div_ts = df.groupby("ex_date")["div_per_share"].sum()
    div_ts = div_ts.reindex(calendar, fill_value=0.0)

    # Trailing 12-month (252 trading days) rolling sum
    trailing_div = div_ts.rolling(252, min_periods=1).sum()

    # Divide by price for yield
    price = close_prices[ticker]
    dy = trailing_div / price
    dy = dy.sub()
    dy = dy.replace([np.inf, -np.inf], np.nan)

    div_yield_d[ticker] = dy

    time.sleep(0.2)

# Dictionaries preserve insertion ordering (e.g. NANO -> NANOV -> TRMED) for multiple name changes.
overlap_fundamentals_merge = {
    "SOHO": "LUMI", 
    "AYFIE": "AIX",
    "TRVX": "CRNA",
    "B2H": "B2I",
    "BRA": "LEA",
    "CSAM": "OMDA",
    "CARBN": "AQUIL", 
    "AQUIL": "DFENS",
    "SIOFF": "SEA1",
    "ELOP": "ININ", 
    "BERGEN": "ENDUR", 
    "ULTI": "ZLNA", 
    "R8P": "RCR", 
    "SBX": "ENH",
    "BGBIO": "ONCIN", 
    "IFISH": "KLDVK",
    "SADG": "ROGS",
    "PSKY": "CODE",
    "HMONY": "LOKO",
    "BEL": "BELCO",
    "NOR": "BNOR",
    "ITE": "ITERA",
    "NOFI": "BANO",
    "INFRNT": "INFRO",
    "SVEG": "SBNOR",
    "NANO": "NANOV",
    "NANOV": "TRMED",
    "SALMON": "SACAM",
    "BDRILL": "BORR",
    "ARCHER": "ARCH",
    "AEGA" : "NOFIN"
    }

# The solution is to overlap old and new tickers, while naming the fundmental column after the most recent ticker symbol.
def overlap_merge(df, merge_map):
    for old, new in merge_map.items():
        if old not in df.columns or new not in df.columns:
            continue
        df[new] = df[new].fillna(df[old])
        df.drop(columns=old, inplace=True)
    return df

# ---------------------------------------------------------
# Output
# ---------------------------------------------------------

div_yield_df = pd.DataFrame(div_yield_d)
div_yield_df = overlap_merge(div_yield_df, overlap_fundamentals_merge)
div_yield_df.to_csv("div_yield.csv")

print(f"Tickers with dividend data: {(div_yield_df > 0).any().sum()}/{len(tickers)}")

In [ ]:
div_yield_df = pd.read_csv("div_yield.csv", index_col=0)


threshold = 0.7
outliers = div_yield_df.stack()[(div_yield_df.stack().abs() > threshold)]
outliers = outliers.reset_index()
outliers.columns = ["date", "ticker", "yield"]
outliers = outliers.sort_values("yield", key=abs, ascending=False)
outliers["yield"] = (outliers["yield"] * 100).round(1).astype(str) + "%"
print(f"Total outlier events: {len(outliers)}")
print(f"Unique tickers: {outliers['ticker'].nunique()}")
print(outliers[["date", "ticker", "yield"]].to_string(index=False))

In [ ]:
print(f"Unique tickers: {outliers['ticker'].unique()}")

In [ ]:
div_yield_df["AKBM"].sort_values(ascending=False)

In [ ]:
close_prices["AKBM"].loc["2024-11-04"]

In [ ]:
adj_close_prices = pd.read_csv("final_adj_close_prices.csv", index_col=0, parse_dates=True)

adj_close_prices["AKBM"].loc["2024-11-04"]

## Momentum and weekly reversal

In [ ]:
close_prices = pd.read_csv("final_adj_close_prices.csv", index_col=0, parse_dates=True)

# Momentum (Relative strength momentum)
relative_stength_momentum = (close_prices.shift(21) / close_prices.shift(252)) - 1 
relative_stength_momentum.dropna(axis=1, how="all")

relative_stength_momentum.to_csv("momentum.csv")

# Reversal
returns = close_prices.pct_change()
weekly_reversal = -returns.rolling(5).sum().shift(1)

weekly_reversal.to_csv("weekly_reversal.csv")

## Growth/investment-rate

In [ ]:
asset_growth_df = pd.read_csv("growth.csv", index_col=0, parse_dates=True)
asset_growth_df.index = pd.to_datetime(asset_growth_df.index)

asset_growth_df

In [ ]:
asset_growth_df.median(axis=0).describe()

In [ ]:
ag_df = asset_growth_df

threshold = 10
outliers = ag_df.stack()[(ag_df.stack().abs() > threshold)]
outliers = outliers.reset_index()
outliers.columns = ["date", "ticker", "growth"]
outliers = outliers.sort_values("growth", key=abs, ascending=False)
outliers["asset_growth_pct"] = (outliers["growth"] * 100).round(1).astype(str) + "%"
print(f"Total outlier events: {len(outliers)}")
print(f"Unique tickers: {outliers['ticker'].nunique()}")
print(outliers[["date", "ticker", "asset_growth_pct"]].to_string(index=False))

## CAPM Idio Vol / Oil Beta / Yield Beta
### Market returns

In [ ]:
def exchange_market_return(api_key, exchange_symbol, start_date, end_date):
    url = f"https://eodhd.com/api/eod/{exchange_symbol}"

    params = {
        "api_token": api_key,
        "fmt": "json",
        "from": start_date,
        "to": end_date
    }

    try:
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print(f"HTTP error, status: {response.status_code}")
                
        data = response.json()

        df = pd.DataFrame(data)
        df["date"] = pd.to_datetime(df["date"])
        df.set_index("date", inplace=True)
        df = df.sort_index() # Fetch adjusted close prices

        return df

    except Exception as e:
        print(f"Error: {type(e).__name__} - {e}")
        return None

exchange_symbol = "OSEBX.INDX"
xosl_prices = exchange_market_return(api_key, exchange_symbol, start_date, end_date)
xosl_prices.to_csv("xosl_prices.csv")
display(xosl_prices)


### Risk-free rate


In [ ]:

# Norwegian central bank yields API: https://data.norges-bank.no/api/data/GOVT_GENERIC_RATES/B.10Y.GBON.?format=sdmx-json&startPeriod=2015-01-01&endPeriod=2026-01-01&locale=en
def get_10yr_historical_yield(start_date, end_date):
    url = f"https://data.norges-bank.no/api/data/GOVT_GENERIC_RATES/B.10Y.GBON.?format=sdmx-json&startPeriod={start_date}&endPeriod={end_date}&locale=en"
    try:
        response = requests.get(url)

        if response.status_code != 200:
            print(f"HTTP error, status: {response.status_code}")
            return None
                
        data = response.json()

        dimensions = data["data"]["structure"]["dimensions"]["observation"]

        time_period_data = next(
            dim for dim in dimensions if dim["id"] == "TIME_PERIOD"
            )
        
        dates = [
            item["id"] for item in time_period_data["values"]
            ]
        
        series_key = list(data["data"]["dataSets"][0]["series"].keys())[0]
        obs_dict = data["data"]["dataSets"][0]["series"][series_key]["observations"]

        yields = [float(obs_dict[str(i)][0]) for i in range(len(dates))]

        df = pd.DataFrame({
            "date": pd.to_datetime(dates),
            "yield": yields
            }).set_index("date")

        return df

    except Exception as e:
        print(f"Error: {type(e).__name__} - {e}")
        return None
    
risk_free = get_10yr_historical_yield(start_date, end_date) / 100
yield10yr_100 = risk_free

risk_free_daily = (1 + risk_free)**(1/252) - 1
risk_free_daily.name = "rf"

In [ ]:
risk_free_daily.to_csv("10year_yield.csv")

In [ ]:

# Norwegian central bank yields API: https://data.norges-bank.no/api/data/GOVT_GENERIC_RATES/B.10Y.GBON.?format=sdmx-json&startPeriod=2015-01-01&endPeriod=2026-01-01&locale=en
def get_3m_historical_yield(start_date, end_date):
    url = f"https://data.norges-bank.no/api/data/GOVT_GENERIC_RATES/B.3M.TBIL.?format=sdmx-json&startPeriod={start_date}&endPeriod={end_date}&locale=en"
    try:
        response = requests.get(url)

        if response.status_code != 200:
            print(f"HTTP error, status: {response.status_code}")
            return None
                
        data = response.json()

        dimensions = data["data"]["structure"]["dimensions"]["observation"]

        time_period_data = next(
            dim for dim in dimensions if dim["id"] == "TIME_PERIOD"
            )
        
        dates = [
            item["id"] for item in time_period_data["values"]
            ]
        
        series_key = list(data["data"]["dataSets"][0]["series"].keys())[0]
        obs_dict = data["data"]["dataSets"][0]["series"][series_key]["observations"]

        yields = [float(obs_dict[str(i)][0]) for i in range(len(dates))]

        df = pd.DataFrame({
            "date": pd.to_datetime(dates),
            "yield": yields
            }).set_index("date")

        return df

    except Exception as e:
        print(f"Error: {type(e).__name__} - {e}")
        return None
    
risk_free = get_3m_historical_yield(start_date, end_date) / 100


risk_free_daily = risk_free)**(1/252) - 1
risk_free_daily.name = "rf"

In [ ]:
risk_free_daily.to_csv("3m_yield.csv")

In [ ]:
def get_nowa_rate(start_date, end_date):
    url = (f"https://data.norges-bank.no/api/data/SHORT_RATES/"
           f"B.NOWA.ON.?format=sdmx-json"
           f"&startPeriod={start_date}&endPeriod={end_date}&locale=en")
    
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()
    
    # Build date index
    dims = data["data"]["structure"]["dimensions"]["observation"]
    time_dim = next(d for d in dims if d["id"] == "TIME_PERIOD")
    dates = [v["id"] for v in time_dim["values"]]
    
    # Pick the rate series: key format is "FREQ:INSTRUMENT:TENOR:UNIT_MEASURE"
    # UNIT_MEASURE index 3 = "R" (Rate). The other indices are T/V/BL/BB.
    series = data["data"]["dataSets"][0]["series"]
    rate_key = next(k for k in series if k.split(":")[3] == "3")
    obs = series[rate_key]["observations"]
    
    yields = [float(obs[str(i)][0]) for i in range(len(dates))]
    
    df = (pd.DataFrame({"date": pd.to_datetime(dates), "rf": yields})
            .set_index("date")
            .sort_index())
    return df

risk_free = get_nowa_rate(start_date, end_date) / 100
risk_free_daily = risk_free / 252
risk_free_daily.columns = ["rf"]

In [ ]:
risk_free_daily.to_csv("NOWA.csv")

### Excess returns calc

In [ ]:
xosl_prices = pd.read_csv("xosl_prices.csv", index_col=0)["adjusted_close"]
xosl_prices.index = pd.to_datetime(xosl_prices.index)

close_prices = pd.read_csv("final_adj_close_prices.csv", index_col=0)
close_prices.index = pd.to_datetime(close_prices.index)

xosl_returns = xosl_prices.pct_change().dropna()
xosl_returns.name = "rm"

returns = close_prices.pct_change().iloc[1:,:]

shared_idx = (returns.index.intersection(xosl_returns.index).intersection(risk_free_daily.index))

returns = returns.loc[shared_idx]
xosl_returns = xosl_returns.loc[shared_idx]
risk_free_daily = risk_free_daily.loc[shared_idx]

# 1. Ensure risk_free_daily is a Series, not a DataFrame with one column
if isinstance(risk_free_daily, pd.DataFrame):
    risk_free_daily = risk_free_daily.iloc[:, 0]

# 2. Reindex the risk-free rate to match the asset returns index
# 'ffill' handles holidays where the bank might have a rate but the exchange is closed
rf_aligned = risk_free_daily.reindex(returns.index, method='ffill')

# 3. Perform subtraction
# If NaNs persist, it means returns itself contains NaNs (common for new/old listings)
excess_returns = returns.sub(rf_aligned, axis=0)
excess_market_returns = xosl_returns.sub(rf_aligned, axis=0)

# 4. Critical Check: Verification of the first valid row
print(f"Total NaNs in Excess Assets: {excess_returns.isna().sum().sum()}")
print(f"Total Rows: {len(excess_returns)}")
excess_returns
# Synchronize indices and handle the unbalanced panel
# We use the XOSL (Market) index as the master trading calendar
common_dates = xosl_returns.index

# Align all components to the market trading days
rf_final = risk_free_daily.reindex(common_dates, method='ffill')
assets_final = returns.reindex(common_dates)
market_final = xosl_returns # already on common_dates

# Calculate Excess Returns
excess_returns = assets_final.sub(rf_final, axis=0)
excess_market_returns = market_final.sub(rf_final, axis=0)

print("Returns NaN:", returns.isna().sum().sum())
print("Excess returns NaN:", excess_returns.isna().sum().sum())
print("Difference:", excess_returns.isna().sum().sum() - returns.isna().sum().sum())

### Rolling Idiosyncratic Volatility and Market Beta

In [ ]:
# Rolling total vol
excess_market_returns = excess_market_returns.squeeze()
total_vol = excess_returns.rolling(60, min_periods=20).std()

# Rolling beta (vectorized, no loop)
cov = excess_returns.rolling(60, min_periods=20).cov(excess_market_returns)
var = excess_market_returns.rolling(60, min_periods=20).var()
beta = cov.div(var, axis=0)

# Idiosyncratic vol = sqrt(total_var - beta^2 * market_var)
idio_var = total_vol**2 - (beta**2).mul(var, axis=0)
idio_var = idio_var.clip(lower=1e-8) # clip
idio_vol = np.sqrt(idio_var)
log_idio_vol = np.log(idio_vol)
problem_tickers = log_idio_vol.columns[log_idio_vol.isin([-np.inf, np.inf]).any()]
print(problem_tickers)

In [ ]:
log_idio_vol.to_csv("log_idio_vol.csv")
beta.to_csv("market_beta.csv")

### Oil Sensitivity/Oil Beta

In [ ]:
rf_aligned = risk_free_daily.reindex(returns.index, method='ffill')

brent_oil = yf.download("BZ=F", start=start_date, end=end_date)
brent_oil_close = brent_oil["Close"]
oil_returns = brent_oil_close.pct_change()
oil_excess_returns = oil_returns.sub(rf_aligned, axis=0)

unified_index = excess_returns.index.union(oil_excess_returns.index)

# Reindex and fill with 0
adj_assets = excess_returns.reindex(unified_index).fillna(np.nan)
adj_oil = oil_excess_returns.reindex(unified_index).fillna(np.nan).squeeze()

# Rolling beta (vectorized, no loop)
cov = adj_assets.rolling(21, min_periods=10).cov(adj_oil)
var = adj_oil.rolling(21, min_periods=10).var()
var = var.clip(lower=1e-6)

oil_rolling_betas = cov.div(var, axis=0)

active = pd.read_csv("shares.csv", index_col=0)
active.index = pd.to_datetime(active.index)

oil_rolling_betas = oil_rolling_betas[active.columns].ffill() # active change

# outlier inspection
threshold = 10.0
outliers = oil_rolling_betas.stack()[oil_rolling_betas.stack().abs() > threshold]
outliers = outliers.reset_index()
outliers.columns = ["date", "ticker", "beta"]
outliers = outliers.sort_values("beta", key=abs, ascending=False)
print(f"Total outlier events: {len(outliers)}")
print(f"Unique tickers: {outliers['ticker'].nunique()}")
print(outliers[["date", "ticker", "beta"]].to_string(index=False))
outliers[["date", "ticker", "beta"]].to_csv("outliers_beta.csv")
outliers_value_counts = outliers[["date", "ticker", "beta"]]["ticker"].value_counts()
outliers_value_counts.to_csv("outliers_value_counts_oil_beta.csv")

In [ ]:
oil_rolling_betas = pd.read_csv("oil_rolling_betas.csv", index_col=0, parse_dates=True)

In [ ]:
oil_rolling_betas.to_csv("oil_rolling_betas.csv")

### Yield spread sensitivity/Yield spread Beta

In [ ]:
# Using changes in the risk-free rate
yield10yr = yield10yr_100 * 100
yield_changes = yield10yr.diff().squeeze()
# yield_changes = risk_free.pct_change().squeeze()
yield_changes.index = pd.to_datetime(yield_changes.index)

unified_index = excess_returns.index.union(yield_changes.index)

# Reindex and fill with 0 (Assumption: No trade = 0 return)
adj_assets = excess_returns.reindex(unified_index).fillna(np.nan)
adj_yield_chg = yield_changes.reindex(unified_index).fillna(np.nan)

cov = adj_assets.rolling(21, min_periods=10).cov(adj_yield_chg)
var = adj_yield_chg.rolling(21, min_periods=10).var()
# var = var.clip(lower=1e-6)

yield_rolling_betas = cov.div(var, axis=0)

active = pd.read_csv("shares.csv", index_col=0)
active.index = pd.to_datetime(active.index)

yield_rolling_betas = yield_rolling_betas[active.columns].ffill() # active change

# Outlier inspection
threshold = 10.0
outliers = yield_rolling_betas.stack()[yield_rolling_betas.stack().abs() > threshold]
outliers = outliers.reset_index()
outliers.columns = ["date", "ticker", "beta"]
outliers = outliers.sort_values("beta", key=abs, ascending=False)
print(f"Total outlier events: {len(outliers)}")
print(f"Unique tickers: {outliers['ticker'].nunique()}")
print(outliers[["date", "ticker", "beta"]].to_string(index=False))
outliers[["date", "ticker", "beta"]].to_csv("outliers_yield_beta.csv")
outliers_value_counts = outliers[["date", "ticker", "beta"]]["ticker"].value_counts()
outliers_value_counts.to_csv("outliers_value_counts_yield_beta.csv")

In [ ]:
yield_rolling_betas.describe()

In [ ]:
yield_rolling_betas.to_csv("yield_rolling_betas.csv")

## Get industry/sector

### GICS Sector

In [ ]:
def get_ticker_gics_sector(api_key, ticker):
    url =  f"https://eodhd.com/api/fundamentals/{ticker}.{exchange}"
    params = {
        "api_token": api_key,
        "fmt": "json",
        "filter": "General"
    }

    response = requests.get(url, params=params)
    if response.status_code != 200:
        return None
    
    data = response.json()
    return data.get("GicSector")

sectors = {}

tickers = pd.read_csv("final_close_prices.csv", index_col=0).columns

for ticker in tqdm(tickers, desc="identifying asset sectors", unit="ticker"):
    print(f"Fetching sector for {ticker}...")
    sector = get_ticker_gics_sector(api_key, ticker)
    if sector:
        sectors[ticker] = sector
    time.sleep(0.1)

df = pd.DataFrame.from_dict(sectors, orient="index", columns=["Sector"])

df.index.name = "Ticker"
df.reset_index(inplace=True)
one_hot = pd.get_dummies(df, columns=["Sector"], dtype=int)
one_hot.to_csv("ticker_gics_sectors_table.csv", index=False)

In [ ]:
gics_sector = pd.read_csv("ticker_gics_sectors_table.csv", index_col=0)
gics_sector.columns = gics_sector.columns.str.replace('^Sector_', '', regex=True)
gics_sector.to_csv("sectors_table.csv")

In [ ]:
gics_sector

In [ ]:
market_cap = pd.read_csv("mcap.csv", index_col=0, parse_dates=True)

gics_sector_exposure_mcap = gics_sector.mul(market_cap.iloc[-1], axis=0).sum() / gics_sector.mul(market_cap.iloc[-1], axis=0).sum().sum()

gics_sector_exposure_mcap

# Information retention

In [ ]:
# ── Load universe from close prices ──────────────────────────────────────────
close_prices = pd.read_csv("final_close_prices.csv", index_col=0, parse_dates=True)
tickers  = close_prices.columns
calendar = pd.to_datetime(close_prices.index)


def fetch_shares_outstanding(api_key, ticker, exchange):
    """
    Fetches the most recent (non-PIT) shares outstanding for a ticker from EODHD.

    Tries SharesStats.SharesOutstanding first, falls back to Highlights.SharesOutstanding.
    Returns a float or None.
    """
    url    = f"https://eodhd.com/api/fundamentals/{ticker}.{exchange}"
    params = {"api_token": api_key, "fmt": "json"}

    try:
        response = requests.get(url, params=params, timeout=30)
        if response.status_code != 200:
            return None

        data   = response.json()
        shares = (data.get("SharesStats", {}) or {}).get("SharesOutstanding")
        if shares is None:
            shares = (data.get("Highlights", {}) or {}).get("SharesOutstanding")
        return float(shares) if shares is not None else None

    except Exception as e:
        print(f"Failed retrieval for {ticker}: {e}")
        return None


# ── Main fetch loop ───────────────────────────────────────────────────────────
shares_raw = {}

for ticker in tqdm(tickers, desc="Fetching shares outstanding"):
    shares_raw[ticker] = fetch_shares_outstanding(api_key, ticker, exchange)
    print(f"Retrieved shares outstanding for {ticker}")
    time.sleep(0.2)

shares_series = pd.Series(shares_raw, name="shares_outstanding")


# ── Overlap merge ─────────────────────────────────────────────────────────────
overlap_fundamentals_merge = {
    "SOHO": "LUMI",
    "AYFIE": "AIX",
    "TRVX": "CRNA",
    "B2H": "B2I",
    "BRA": "LEA",
    "CSAM": "OMDA",
    "CARBN": "AQUIL",
    "AQUIL": "DFENS",
    "SIOFF": "SEA1",
    "ELOP": "ININ",
    "BERGEN": "ENDUR",
    "ULTI": "ZLNA",
    "R8P": "RCR",
    "SBX": "ENH",
    "BGBIO": "ONCIN",
    "IFISH": "KLDVK",
    "SADG": "ROGS",
    "PSKY": "CODE",
    "HMONY": "LOKO",
    "BEL": "BELCO",
    "NOR": "BNOR",
    "ITE": "ITERA",
    "NOFI": "BANO",
    "INFRNT": "INFRO",
    "SVEG": "SBNOR",
    "NANO": "NANOV",
    "NANOV": "TRMED",
    "SALMON": "SACAM",
    "BDRILL": "BORR",
    "ARCHER": "ARCH",
    "AEGA": "NOFIN"
}

for old, new in overlap_fundamentals_merge.items():
    if old not in shares_series.index:
        continue
    if new in shares_series.index and pd.isna(shares_series[new]):
        shares_series[new] = shares_series[old]
    shares_series = shares_series.drop(index=old)

print(f"Tickers after overlap merge: {shares_series.notna().sum()}/{len(shares_series)}")


# ── Output ────────────────────────────────────────────────────────────────────
shares_series.to_csv("shares_outstanding_(NOT PIT).csv", header=True)

missing = shares_series[shares_series.isna()]
if len(missing) > 0:
    print(f"\nMissing shares outstanding ({len(missing)}):")
    for t in missing.index:
        print(f"  {t}")
    missing.reset_index().rename(columns={"index": "ticker"}).to_csv(
        "shares_outstanding_missing.csv", index=False
    )

print("\nRetrieval complete")
print(f"Tickers with data: {shares_series.notna().sum()}/{len(shares_series)}")

# Inspect final data

## Removed tickers
Tickers were removed due to:
- Lacking feature data.
- Lacking the ability to be adjusted to for correct point-in-time arrival of information.
- Broken price data
- Broken feature data.

In [ ]:
original_tickers = pd.read_csv("final_adj_close_prices.csv", index_col=0, parse_dates=True).columns
original_tickers = set(original_tickers)

size = pd.read_csv("log_mcap(size).csv", index_col=0, parse_dates=True)

tickers = set(size.dropna(axis=1, how="all").columns)

missing_tickers = list(original_tickers - tickers)

len(missing_tickers)

# Final price  series cleaning

In [ ]:
## Identify outliers/Faulty data
### close
adj_close_prices = pd.read_csv("final_adj_close_prices.csv", index_col=0, parse_dates=True)

returns = adj_close_prices.pct_change()

threshold = 2.0
outliers = returns.stack()[returns.stack().abs() > threshold]
outliers = outliers.reset_index()
outliers.columns = ["date", "ticker", "return"]
outliers = outliers.sort_values("return", key=abs, ascending=False)
outliers["return_pct"] = (outliers["return"] * 100).round(1).astype(str) + "%"
print(f"Total outlier events: {len(outliers)}")
print(f"Unique tickers: {outliers['ticker'].nunique()}")
print(outliers[["date", "ticker", "return_pct"]].to_string(index=False))
outliers[["date", "ticker", "return_pct"]].to_csv("outliers_returns.csv")

In [ ]:

repeat_offenders = outliers.groupby("ticker").filter(lambda x: len(x) >= 2)
print(repeat_offenders["ticker"].value_counts())
print(repeat_offenders[["date", "ticker", "return_pct"]].to_string(index=False))

We drop NOFIN and AEGA as well, since the price data is corrupted.

In [ ]:
overlap_fundamentals_merge = {
    "SOHO": "LUMI", 
    "AYFIE": "AIX",
    "TRVX": "CRNA",
    "B2H": "B2I",
    "BRA": "LEA",
    "CSAM": "OMDA",
    "CARBN": "AQUIL", 
    "AQUIL": "DFENS",
    "SIOFF": "SEA1",
    "ELOP": "ININ", 
    "BERGEN": "ENDUR", 
    "ULTI": "ZLNA", 
    "R8P": "RCR", 
    "SBX": "ENH",
    "SRBNK": "SB1NO",
    "BGBIO": "ONCIN", 
    "IFISH": "KLDVK",
    "SADG": "ROGS",
    "PSKY": "CODE",
    "HMONY": "LOKO",
    "BEL": "BELCO",
    "NOR": "BNOR",
    "ITE": "ITERA",
    "NOFI": "BANO",
    "INFRNT": "INFRO",
    "SVEG": "SBNOR",
    "NANO": "NANOV",
    "NANOV": "TRMED",
    "SALMON": "SACAM",
    "BDRILL": "BORR",
    "ARCHER": "ARCH",
    "AEGA" : "NOFIN"
    }

# Tickers with illegitimate price moves
illeg_tickers = ["KMC", "ASA", "CRNA", "DFENS", "SEAW7", "NODL", "GEM", "NOFIN", "KMCP", "ONCIN"]

remove = list(overlap_fundamentals_merge.keys())
remove.extend(illeg_tickers)
remove = set(remove)
close_prices = pd.read_csv("final_close_prices.csv", index_col=0, parse_dates=True)
close_prices = close_prices.drop(columns=remove, errors="ignore")
adj_close_prices = pd.read_csv("final_adj_close_prices.csv", index_col=0, parse_dates=True)
adj_close_prices = adj_close_prices.drop(columns=remove, errors="ignore")

book = pd.read_csv("book_equity.csv", index_col=0).dropna(how="all", axis=1)
div = pd.read_csv("div_yield.csv", index_col=0)

# Common tickers across all three
common = book.columns.intersection(div.columns).intersection(close_prices.columns).intersection(adj_close_prices.columns)
print(f"Book: {len(book.columns)}, Div: {len(div.columns)}, Prices: {len(close_prices.columns)}")
print(f"Common: {len(common)}")

# Show what's missing from each
print(f"\nIn div but not prices: {sorted(set(div.columns) - set(close_prices.columns))}")
print(f"In prices but not div: {sorted(set(close_prices.columns) - set(div.columns))}")
print(f"In prices but not book: {sorted(set(close_prices.columns) - set(book.columns))}")

# Align all to common tickers
# Drop all shared NaN rows
close_prices = close_prices[common].dropna(how="all", axis=0)
adj_close_prices = adj_close_prices[common].dropna(how="all", axis=0)

close_prices.to_csv("final_close_prices_aligned.csv")
adj_close_prices.to_csv("final_adj_close_prices_aligned.csv")

In [ ]:
# Load the aligned universe
common = pd.read_csv("final_adj_close_prices_aligned.csv", index_col=0).columns

# Align all characteristics
for filename in [
    "book-to-market(value).csv",
    "log_mcap(size).csv",
    "illiquidity.csv",
    "dtb(leverage).csv", 
    "div_yield.csv",
    "momentum.csv",
    "weekly_reversal.csv",
    "growth.csv",
    "log_idio_vol.csv",
    "oil_rolling_betas.csv",
    "yield_rolling_betas.csv",
    "volume.csv",
    "market_beta.csv",
    "mcap.csv"
]:
    df = pd.read_csv(filename, index_col=0)
    # Keep only common tickers, add missing as NaN
    df = df.reindex(columns=common)
    name = filename.replace(".csv", "")
    df.to_csv(f"{name}_aligned_final_2.csv")
    present = df.notna().any().sum()
    print(f"{filename}: {present}/{len(common)} tickers with data")

In [ ]:
## Identify outliers/Faulty data
### close
close_prices = pd.read_csv("final_close_prices_aligned.csv", index_col=0, parse_dates=True)

returns = adj_close_prices.pct_change()

threshold = 1.0
outliers = returns.stack()[returns.stack().abs() > threshold]
outliers = outliers.reset_index()
outliers.columns = ["date", "ticker", "return"]
outliers = outliers.sort_values("return", key=abs, ascending=False)
outliers["return_pct"] = (outliers["return"] * 100).round(1).astype(str) + "%"
print(f"Total outlier events: {len(outliers)}")
print(f"Unique tickers: {outliers['ticker'].nunique()}")
print(outliers[["date", "ticker", "return_pct"]].to_string(index=False))
outliers[["date", "ticker", "return_pct"]].to_csv("outliers_returns.csv")

In [ ]:
## Identify outliers/Faulty data
### close
adj_close_prices = pd.read_csv("final_adj_close_prices_aligned.csv", index_col=0, parse_dates=True)

returns = adj_close_prices.pct_change()

threshold = 1.0
outliers = returns.stack()[returns.stack().abs() > threshold]
outliers = outliers.reset_index()
outliers.columns = ["date", "ticker", "return"]
outliers = outliers.sort_values("return", key=abs, ascending=False)
outliers["return_pct"] = (outliers["return"] * 100).round(1).astype(str) + "%"
print(f"Total outlier events: {len(outliers)}")
print(f"Unique tickers: {outliers['ticker'].nunique()}")
print(outliers[["date", "ticker", "return_pct"]].to_string(index=False))
outliers[["date", "ticker", "return_pct"]].to_csv("outliers_returns.csv")

In [ ]:
repeat_offenders = outliers.groupby("ticker").filter(lambda x: len(x) >= 1)
print(repeat_offenders["ticker"].value_counts())
print(repeat_offenders[["date", "ticker", "return_pct"]].to_string(index=False))

Tickers with illegitimate extreme price moves
- KMC
- ASA
- CRNA
- DFENS
- SEAW7
- NODL
- GEM

In [ ]:
yield_rolling_betas = yield_rolling_betas[adj_close_prices.columns.to_list()].drop(columns = ["PRS", "ENERG"])

# Outlier inspection
threshold = 10.0
outliers = yield_rolling_betas.stack()[yield_rolling_betas.stack().abs() > threshold]
outliers = outliers.reset_index()
outliers.columns = ["date", "ticker", "beta"]
outliers = outliers.sort_values("beta", key=abs, ascending=False)
print(f"Total outlier events: {len(outliers)}")
print(f"Unique tickers: {outliers['ticker'].nunique()}")
print(outliers[["date", "ticker", "beta"]].to_string(index=False))
outliers[["date", "ticker", "beta"]].to_csv("outliers_yield_beta.csv")
outliers_value_counts = outliers[["date", "ticker", "beta"]]["ticker"].value_counts()
outliers_value_counts.to_csv("outliers_value_counts_yield_beta.csv")